In [0]:
from pyspark.sql.functions import (
    col, sum, count, countDistinct, avg, max, min,
    when, coalesce, lit, round
)

silver_base = "abfss://silver@ecommercenidhi.dfs.core.windows.net"

customers = spark.read.format("delta").load(f"{silver_base}/customers")
products = spark.read.format("delta").load(f"{silver_base}/products")
orders = spark.read.format("delta").load(f"{silver_base}/orders")
payments = spark.read.format("delta").load(f"{silver_base}/payments")
events = spark.read.format("delta").load(f"{silver_base}/website_events")

print("Customers:", customers.count())
print("Products:", products.count())
print("Orders:", orders.count())
print("Payments:", payments.count())
print("Events:", events.count())

In [0]:
display(
    orders.groupBy("status")
    .count()
    .orderBy("status")
)

In [0]:
sales_daily = (
    orders
    .groupBy("order_date")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        round(sum("amount"), 2).alias("total_revenue"),
        round(avg("amount"), 2).alias("average_order_value")
    )
    .orderBy("order_date")
)

display(sales_daily)

In [0]:
gold_base = "abfss://gold@ecommercenidhi.dfs.core.windows.net"

sales_daily.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_base}/sales_daily")

In [0]:
product_performance = (
    orders
    .join(
        products,
        orders.product_id == products.product_id,
        "left"
    )
    .groupBy(
        products.product_id,
        products.product_name,
        products.category
    )
    .agg(
        countDistinct(orders.order_id).alias("total_orders"),
        sum(orders.quantity).alias("units_sold"),
        round(sum(orders.amount), 2).alias("total_revenue"),
        round(avg(orders.amount), 2).alias("average_order_value")
    )
    .orderBy(col("total_revenue").desc())
)

display(product_performance)

In [0]:
product_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_base}/product_performance")

In [0]:
revenue_by_city = (
    orders
    .join(
        customers,
        orders.customer_id == customers.customer_id,
        "left"
    )
    .groupBy("city")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        round(sum("amount"), 2).alias("total_revenue"),
        round(avg("amount"), 2).alias("average_order_value")
    )
    .orderBy(col("total_revenue").desc())
)

display(revenue_by_city)

In [0]:
revenue_by_city.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_base}/revenue_by_city")

In [0]:
customer_orders = (
    orders
    .groupBy("customer_id")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        round(sum("amount"), 2).alias("total_spend"),
        round(avg("amount"), 2).alias("average_order_value"),
        max("order_date").alias("last_order_date")
    )
)

In [0]:
customer_events = (
    events
    .groupBy("customer_id")
    .agg(
        countDistinct("event_id").alias("total_events"),
        countDistinct("event_type").alias("unique_event_types")
    )
)

In [0]:
customer_360 = (
    customers
    .join(customer_orders, "customer_id", "left")
    .join(customer_events, "customer_id", "left")
    .select(
        "customer_id",
        "name",
        "email",
        "city",
        "country",
        "signup_date",
        coalesce(col("total_orders"), lit(0)).alias("total_orders"),
        coalesce(col("total_spend"), lit(0.0)).alias("total_spend"),
        coalesce(col("average_order_value"), lit(0.0)).alias("average_order_value"),
        "last_order_date",
        coalesce(col("total_events"), lit(0)).alias("total_events"),
        coalesce(col("unique_event_types"), lit(0)).alias("unique_event_types")
    )
)

display(customer_360)

In [0]:
customer_360.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_base}/customer_360")

In [0]:
order_metrics = (
    orders
    .join(
        customers.select(
            "customer_id",
            "name",
            "city"
        ),
        "customer_id",
        "left"
    )
    .join(
        products.select(
            "product_id",
            "product_name",
            "category"
        ),
        "product_id",
        "left"
    )
    .select(
        "order_id",
        "order_date",
        "customer_id",
        "name",
        "city",
        "product_id",
        "product_name",
        "category",
        "quantity",
        "amount",
        "status"
    )
)

display(order_metrics)

In [0]:
order_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_base}/order_metrics")

In [0]:
gold_tables = [
    "sales_daily",
    "product_performance",
    "revenue_by_city",
    "customer_360",
    "order_metrics"
]

for table in gold_tables:
    path = f"{gold_base}/{table}"
    df = spark.read.format("delta").load(path)
    print(f"{table}: {df.count()} rows")

In [0]:
display(
    dbutils.fs.ls(
        "abfss://bronze@ecommercenidhi.dfs.core.windows.net/incremental/"
    )
)